# QMIX Monolithic Training
This notebook runs the Monolithic QMIX baseline training on Google Colab using CPU only.

**Requirements:**
- Google Colab account
- Project files uploaded to Google Drive or cloned from repository
- SUMO traffic simulator (will be installed automatically)

**Training Setup:**
- Algorithm: QMIX (Monotonic Value Function Factorization)
- Environment: SUMO Traffic Simulation
- Device: CPU (compatible with local machine setup)
- Max Agents: 50 vehicles
- Episodes: 100 (configurable)

## 1. Environment Setup

First, we'll install SUMO (traffic simulator) and required dependencies.

In [1]:
# Check if running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab")
else:
    print("Running locally")

In [2]:
# Install SUMO on Colab (Ubuntu-based)
if IN_COLAB:
    print("Installing SUMO...")
    !apt-get update -qq
    !apt-get install -y -qq sumo sumo-tools sumo-doc
    
    # Set SUMO_HOME environment variable
    import os
    os.environ['SUMO_HOME'] = '/usr/share/sumo'
    
    print("SUMO installed successfully!")
    !sumo --version
else:
    print("Skipping SUMO installation (assuming already installed locally)")

In [3]:
# Install Python dependencies
if IN_COLAB:
    print("Installing Python packages...")
    %pip install -q torch numpy matplotlib sumolib traci pyyaml tensorboard
    print("Dependencies installed!")
else:
    print("Skipping package installation (assuming dependencies already installed)")

## 2. Get Project Files

**Recommended: Clone from GitHub**
- Automatically downloads the latest version from GitHub
- No need to upload files manually

**Alternative: Use Google Drive**
- Useful if you have local modifications to test
- Requires manual upload to Drive

In [4]:
# Option A: Clone from GitHub (Recommended)
if IN_COLAB:
    print("Cloning project from GitHub...")
    !git clone https://github.com/Akiri017/apriori_civiq.git
    
    import os
    PROJECT_PATH = '/content/apriori_civiq'
    os.chdir(PROJECT_PATH)
    print(f"Project cloned successfully to: {PROJECT_PATH}")
    print(f"Working directory: {os.getcwd()}")
else:
    import os
    cwd = os.getcwd()
    if os.path.basename(cwd) == 'notebooks':
        PROJECT_PATH = os.path.dirname(cwd)
    else:
        PROJECT_PATH = cwd
    print(f"Working directory: {PROJECT_PATH}")

In [ ]:
# Option B: Mount Google Drive (alternative - uncomment if needed)
# if IN_COLAB:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     
#     # Update this path to your Drive folder
#     PROJECT_PATH = '/content/drive/MyDrive/apriori_civiq'
#     
#     import os
#     if os.path.exists(PROJECT_PATH):
#         os.chdir(PROJECT_PATH)
#         print(f"Working directory: {os.getcwd()}")
#     else:
#         print(f"ERROR: Path not found: {PROJECT_PATH}")
#         print("Update PROJECT_PATH to match your Google Drive folder structure")

## 3. Verify Project Structure

Let's verify that all necessary files are present.

In [5]:
import os

# Check key files and directories
required_paths = [
    'src/main.py',
    'src/baseline/mono_qmix/',
    'scenarios/test_simple/',
]

print("Checking project structure...")
print(f"Current directory: {os.getcwd()}\n")

missing = []
for path in required_paths:
    full_path = os.path.join(PROJECT_PATH, path)
    exists = os.path.exists(full_path)
    status = "FOUND" if exists else "MISSING"
    print(f"{status} {path}")
    if not exists:
        missing.append(path)

if missing:
    print(f"\nWarning: Missing {len(missing)} required paths")
    print("\nTroubleshooting:")
    print("1. Verify PROJECT_PATH is correct in the previous cell")
    print("2. Make sure you uploaded the entire project folder to Google Drive")
    print("3. Check that the folder contains the 'src' and 'scenarios' directories")
else:
    print("\nAll required files found.")

## 4. Configure Training Parameters

Modify these parameters based on your needs. For Colab, we use CPU-only and disable GUI.

In [6]:
# Training Configuration
TRAINING_CONFIG = {
    # Environment
    'sumo_config': os.path.join(PROJECT_PATH, 'scenarios/test_simple/Configuration_med.sumocfg'),
    'max_agents': 20,
    'use_gui': False,  # False for Colab (no display)
    
    # Training hyperparameters
    'episodes': 50,  # Reduced for initial testing (was 100)
    'batch_size': 16,  # Reduced for stability (was 32)
    'buffer_capacity': 2000,  # Reduced for memory (was 5000)
    'max_episode_length': 500,  # Reduced for stability (was 3600)
    
    # Learning parameters
    'learning_rate': 0.0005,
    'gamma': 0.99,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay': 0.995,
    
    # Updates
    'target_update_freq': 200,
    'min_buffer_size': 64,  # Reduced (was 128)
    
    # Device
    'device': 'cpu',  # CPU-only for consistency with local training
    
    # Monitoring
    'verbose': True,  # Print detailed step information
    'step_interval': 50,  # Print status every N steps during episode
}

print("Training Configuration (Optimized for Colab):")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")
print("\nNote: Config optimized for stability on Colab")
print("   Increase episodes/agents/length after successful test run")

## 5. Import Required Modules

Import the QMIX implementation and supporting libraries.

In [ ]:
# Option A: Clone from GitHub (Recommended)
if IN_COLAB:
    print("Cloning project from GitHub...")
    !git clone https://github.com/Akiri017/apriori_civiq.git
    
    import os
    PROJECT_PATH = '/content/apriori_civiq'
    os.chdir(PROJECT_PATH)
    print(f"Project cloned successfully to: {PROJECT_PATH}")
    print(f"Working directory: {os.getcwd()}")
else:
    import os
    # FIX: When opened locally in VS Code, the notebook kernel's cwd is the
    # notebooks/ subfolder, not the project root. This caused sys.path to point
    # at notebooks/src/ (nonexistent) instead of the actual src/ directory.
    # BEFORE: PROJECT_PATH = os.getcwd()
    # NOW: Walk up one level if cwd is the notebooks/ folder.
    cwd = os.getcwd()
    if os.path.basename(cwd) == 'notebooks':
        PROJECT_PATH = os.path.dirname(cwd)
    else:
        PROJECT_PATH = cwd
    print(f"Working directory: {PROJECT_PATH}")

# Add src to path
import sys
import os
sys.path.insert(0, os.path.join(PROJECT_PATH, 'src'))

# Import training modules
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
from datetime import datetime

from baseline.mono_qmix import QMIX_Controller, QMIXSumoEnv, EpisodeReplayBuffer

print("Modules imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {TRAINING_CONFIG['device']}")

## 6. Define Training Function

This is adapted from the main training script, optimized for notebook execution.

In [ ]:
# Quick SUMO test to catch configuration errors early
print("Testing SUMO environment setup...")

# First, close any existing SUMO connections
try:
    import traci
    if traci.isLoaded():
        print("Closing existing SUMO connection...")
        traci.close()
        print("Previous connection closed")
except:
    pass

try:
    # Create environment (this may start SUMO in __init__)
    test_env = QMIXSumoEnv(
        sumo_config=TRAINING_CONFIG['sumo_config'],
        max_agents=TRAINING_CONFIG['max_agents'],
        use_gui=False
    )
    
    print("SUMO initialized successfully")
    
    # Close and reset to test the full cycle
    print("Testing environment reset cycle...")
    test_env.close()
    
    state, obs = test_env.reset()
    print(f"State shape: {state.shape}")
    print(f"Observation shape: {obs.shape}")
    
    print("Testing 10 simulation steps...")
    for i in range(10):
        # Random actions for all agents
        actions = np.random.randint(0, test_env.n_actions, size=test_env.max_agents)
        try:
            next_state, next_obs, reward, done = test_env.step(actions)
            if i % 5 == 0:
                print(f"  Step {i+1}: reward={reward:.2f}, done={done}")
        except Exception as e:
            print(f"Error at step {i+1}: {e}")
            break
    
    print("Test steps completed")
    
    # Clean up
    test_env.close()
    print("Test environment closed")
    print("\nSUMO test passed! Ready for training.")
    
except Exception as e:
    print(f"\nSUMO test failed: {e}")
    print("\nTroubleshooting:")
    print("1. Run the 'Close Stray SUMO Connections' cell above")
    print("2. Check that the SUMO config file exists and is valid")
    print("3. Verify SUMO_HOME is set correctly")
    print("4. Restart the kernel if connection issues persist")
    import traceback
    traceback.print_exc()
    
    # Attempt cleanup
    try:
        import traci
        if traci.isLoaded():
            traci.close()
            print("\nCleaned up SUMO connection")
    except:
        pass

## 7. Run Training

Execute the training process. This may take a while depending on the number of episodes.

In [ ]:
# Run training
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

results = train_qmix(TRAINING_CONFIG)

print(f"\nEnd time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("Training completed!")

## 10. Download Results (Optional)

If you want to download the models to your local machine, use the code below.

In [ ]:
# Download files to local machine (Colab only)
if IN_COLAB:
    from google.colab import files
    
    print("Download trained models?")
    print("Uncomment the lines below to download:")
    
    # Uncomment to download
    # files.download(agent_path)
    # files.download(mixer_path)
    # files.download(history_path)
    # files.download(config_path)
else:
    print("Running locally - files already saved to disk")

## 11. Test Trained Model (Optional)

Run a test episode with the trained model to see its performance.

In [ ]:
# Test the trained model
def test_model(controller, config, num_episodes=5):
    """Run test episodes with trained model."""
    print("\nTesting trained model...")
    
    # Initialize environment (no GUI)
    env = QMIXSumoEnv(
        sumo_config=config['sumo_config'],
        max_agents=config['max_agents'],
        use_gui=False
    )
    
    test_rewards = []
    
    try:
        for ep in range(num_episodes):
            state, obs = env.reset()
            hidden = controller.init_hidden(batch_size=1)
            
            episode_reward = 0
            done = False
            t = 0
            
            # Run episode with greedy policy (epsilon=0)
            while not done and t < config['max_episode_length']:
                actions, hidden = controller.select_actions(obs, hidden, epsilon=0.0)
                next_state, next_obs, reward, done = env.step(actions)
                
                obs = next_obs
                state = next_state
                episode_reward += reward
                t += 1
            
            test_rewards.append(episode_reward)
            print(f"  Test Episode {ep+1}/{num_episodes}: Reward = {episode_reward:.2f}, Steps = {t}")
    
    finally:
        env.close()
    
    print(f"\nTest Results:")
    print(f"  Mean Reward: {np.mean(test_rewards):.2f} ± {np.std(test_rewards):.2f}")
    print(f"  Min Reward:  {np.min(test_rewards):.2f}")
    print(f"  Max Reward:  {np.max(test_rewards):.2f}")
    
    return test_rewards

# Uncomment to run tests
# if len(results['rewards_history']) > 0:
#     test_rewards = test_model(results['controller'], TRAINING_CONFIG)

## Summary

This notebook provides a complete pipeline for training the Monolithic QMIX baseline on Google Colab using CPU.

**Key Features:**
- ✓ CPU-only training (compatible with local setup)
- ✓ Automatic SUMO installation
- ✓ GitHub integration (auto-clone latest version)
- ✓ **Verbose logging** - See step-by-step progress
- ✓ **Error recovery** - Automatic SUMO restart on crashes
- ✓ **Progress monitoring** - Real-time status updates during episodes
- ✓ **Pre-flight test** - Validate SUMO before training
- ✓ Model checkpointing and visualization

**Recommended Workflow:**
1. Run all setup cells (1-5)
2. **Run the SUMO test cell (5.5)** to catch issues early
3. Adjust `TRAINING_CONFIG` if test fails (see Troubleshooting section)
4. Run training (section 7)
5. Monitor verbose output to track progress
6. Visualize results and save models

**Performance Tips:**
- Start with smaller episodes (500-1000 steps) and fewer agents (20-30)
- Enable verbose mode to see what's happening
- Use the troubleshooting section if you encounter errors
- Gradually increase complexity after successful initial runs

**Next Steps:**
1. Experiment with different hyperparameters
2. Try different traffic scenarios (change `sumo_config`)
3. Increase episodes/complexity after validation
4. Compare with other baselines or hierarchical approaches

## Troubleshooting SUMO Errors

**Common issues and solutions:**

### 1. "Retrying in 1 seconds" or TraCI communication errors
**Cause:** SUMO crashed or became unresponsive during simulation

**Solutions:**
- ✓ Enable `verbose=True` in training config to see progress
- ✓ Reduce `max_episode_length` (try 1000 instead of 3600)
- ✓ Use simpler scenario (`test_simple` instead of `bgc_full`)
- ✓ Reduce `max_agents` (try 20-30 instead of 50)
- ✓ Run the SUMO test cell first to catch config issues

### 2. "unpack requires a buffer" error
**Cause:** SUMO sent corrupted data, usually means it crashed

**Solutions:**
- Check SUMO scenario files are valid (`.sumocfg`, `.net.xml`, `.rou.xml`)
- Ensure routes don't have impossible paths
- Try running SUMO manually: `sumo -c scenarios/test_simple/Configuration_med.sumocfg --step-length 1`

### 3. Training is slow or hangs
**Cause:** Long episodes with many vehicles

**Solutions:**
- Reduce `max_episode_length` to 500-1000 steps
- Reduce `max_agents` to 20-30
- Set `step_interval=50` to see progress more frequently
- Use smaller traffic scenario

### 4. Out of memory errors
**Solutions:**
- Reduce `buffer_capacity` to 1000-2000
- Reduce `batch_size` to 16
- Reduce `max_episode_length` to 500
- Reduce `max_agents` to 20

### Quick Fix Configuration (if having issues):
```python
TRAINING_CONFIG = {
    'sumo_config': 'scenarios/test_simple/Configuration_med.sumocfg',
    'max_agents': 20,  # Reduced from 50
    'use_gui': False,
    'episodes': 50,  # Reduced from 100
    'batch_size': 16,  # Reduced from 32
    'buffer_capacity': 2000,  # Reduced from 5000
    'max_episode_length': 500,  # Reduced from 3600
    'learning_rate': 0.0005,
    'gamma': 0.99,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay': 0.995,
    'target_update_freq': 200,
    'min_buffer_size': 64,  # Reduced from 128
    'device': 'cpu',
    'verbose': True,  # Enable detailed logging
    'step_interval': 50,  # Status update every 50 steps
}
```